# **Pump controls and hydraulic performance**

**Engineering question:** How do pump control rules affect tank levels, pump switching, and system pressures?

We will use the Net3 hydraulic model. First, we will inspect the network and its existing controls, run a baseline simulation, and visualize hydraulic results. Then we will change the tank-level control thresholds for pump 335 and compare the results. Finally, you will investigate the operational tradeoffs in pairs.

## **1. Load and visualize the network**

In [ ]:
try:
    import wntr
except ImportError:
    %pip install wntr
    import wntr
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
inp = 'Net3_controls.inp'  # adjust path if needed
wn = wntr.network.WaterNetworkModel(inp)

In [ ]:
# get simulation duration
wn.options.time.duration/3600

In [ ]:
# set simulation duration to 2 days
wn.options.time.duration = 💡

In [ ]:
# plot the network
wntr.graphics.plot_network(wn, title='Net3 network', node_size=20, link_width=1)

In [ ]:
# get tanks, pumps
print('Tanks:', wn.tank_name_list)
print('Pumps:', wn.pump_name_list)


In [ ]:
# let's inspect our pumps and tanks
tank = wn.get_node(wn.tank_name_list[💡])


In [ ]:
tank


In [ ]:
# let's inspect our pumps
pump = wn.get_link(wn.pump_name_list[💡])
pump

In [ ]:
# we will use unit conversion later
units = 15850.3  # m3/s to gpm
FT2M = 0.3048


In [ ]:
# run simulation
sim = wntr.sim.EpanetSimulator(wn)
💡  = sim.run_sim()           # save base simulation results
time_hours = results.node['pressure'].index / 3600

## **2. Visualize baseline hydraulic performance**

### **Temporal plots**: tank level, pump flow, and pump status

In [ ]:
tank_level = results.node['pressure']['1'] / FT2M   # ft above tank bottom
pump_flow = results.link['flowrate']['335'] * units # gpm units

plt.plot(time_hours, tank_level)
plt.xlabel('Time (hours)')
plt.ylabel('Tank level (ft)')
plt.title('Tank 1 water level')
plt.show()

plt.plot(time_hours, pump_flow)
plt.xlabel('Time (hours)')
plt.ylabel('Pump flow (gpm)')
plt.title('Pump 335 flow')
plt.show()


### **Spatial plots**: pressure distribution at two times

In [ ]:
print('Available times (hours):', time_hours[:10])

pressure_0 = results.node['pressure'].loc[0]
pressure_24 = results.node['pressure'].loc[24*3600]

wntr.graphics.plot_network(wn, node_attribute=pressure_0,
                           node_size=35, link_width=1,
                           title='Pressure head at 0 hours (m)',node_cmap='coolwarm',node_colorbar_label='Pressure (m)',node_range=[0, 80])
wntr.graphics.plot_network(wn, node_attribute=pressure_24,
                           node_size=35, link_width=1,
                           title='Pressure head at 24 hours (m)',node_cmap='coolwarm',node_colorbar_label='Pressure (m)',node_range=[0, 80])

## **3. Two simple performance indicators**

### A. Count pump switching events (explore on your own)

A switching event occurs whenever the pump status changes between consecutive simulation results.

Python treats `True` and `False` as numbers (`True == 1` and `False == 0`) that's why `sum(I)` gives you the number of times the condition is true

Plot the pump’s ON/OFF status over time and report how many times the pump turns ON.

In [ ]:
I = pump_flow > 0
# print(I)

In [ ]:
print(f'Number of time steps the pump is ON = {sum(I)}')
print(f'Number of time steps the pump is OFF = {sum(~I)}')

In [ ]:
# plot

plt.plot(time_hours, I, color='red', linewidth = 2, alpha = 0.5)
plt.title(f'Pump {pump.name} is ON/OFF')
plt.xlabel('Time (hours)')
plt.show()

In [ ]:
# Convert True/False to 1/0
I = I.astype(int)

# Calculate changes in pump status between consecutive time steps
change = I.diff()

# Take the absolute value to count both ON and OFF switches
switch = change.abs()

# Count the total number of switches
switches = switch.sum()

print('Number of pump switches:', switches)

### B. Count high-pressure time steps at each junction (explore on your own)

In [ ]:
# Get pressure at all junctions and convert from meters to psi
pressure = results.node['pressure'][wn.junction_name_list]
pressure_psi = pressure / FT2M / 2.31

# Identify time steps when pressure exceeds 70 psi
I = pressure_psi >= 70

# Count exceedances at each junction
high_pressure = I.sum()

# Plot the distribution across junctions
plt.hist(high_pressure, bins=15)
plt.xlabel('Number of time steps with pressure >= 70 psi')
plt.ylabel('Number of junctions')
plt.title('High-pressure exceedances')
plt.show()


## **4. Inspect existing controls and run the baseline**
The `wntr.network.controls` module enables setting up, modifying, removing and adding controls.

There are two types of control:
* **Event-based**
* **Time-based**

The general process for setting up controls is to define:
1. Actions (the actuator) `ControlAction`
2. Condition (the trigger for the event) `ControlCondition`
3. Control (actions + conditions) `Control`
4. Add/Remove the control `remove_control`, `add_control`

For example: `IF TANK 1 LEVEL ABOVE 5 THEN PIPE 330 STATUS IS OPEN`. Here:
* Action - pump 330 open
* Condition - tank 1 above 5
* Control - if tank 1 above 5 then pump 330 open

In [ ]:
# we are doing this just so that we can write controls instead of wntr.network.controls
import wntr.network.controls as controls
from wntr.network.controls import LinkStatus

In [ ]:
# Print all current controls
for name, control in wn.controls():
    print(name, control)

**Look at the network:** Where is Tank 1? Find pump 335 and bypass pipe 330. Why might a bypass be needed?

The original notebook uses tank-level thresholds to control pump **335** and bypass **330**. Inspect the four controls associated with these links before changing anything. Check with EPANET.

We will modify the pump and bypass controls from the original notebook. First, predict what happens if the difference between the low and high tank-level thresholds becomes larger.

## **5. Change event-based controls**
## Let's modify the last four controls
* `Pump 335 controlled by level in Tank 1`
* `When pump is closed, bypass pipe is opened`
* Link 335 OPEN IF Node 1 BELOW 17.1 (original) ---> 16.5 (new)
* Link 335 CLOSED IF Node 1 ABOVE 19.1 (original) ---> 22.5 (new)
* Link 330 CLOSED IF Node 1 BELOW 17.1 (original) ---> 16.5 (new)
* Link 330 OPEN IF Node 1 ABOVE 19.1 (original) ---> 22.5 (new)
* Note this is in ft

In [ ]:
# get the controlled elements
tank = wn.get_node('1')
pump = wn.get_link('335')
bypass = wn.get_link('330')


In [ ]:
# Define new tank-level thresholds (ft)
low_level = 💡 *FT2M     # convert units to m
high_level = 💡 *FT2M

# Pump actions
open_action = controls.ControlAction(pump, 'status', LinkStatus.Open)
close_action = controls.ControlAction(pump, 'status', LinkStatus.Closed)

# Tank-level conditions
low_cond = controls.ValueCondition(tank, 'level', '<', 💡 )
high_cond = controls.ValueCondition(tank, 'level', '>', 💡 )

# Pump controls
open_ctrl = controls.Control(💡 , 💡 )
close_ctrl = controls.Control(💡 , 💡 )

wn.remove_control('control 3')
wn.remove_control('control 4')
wn.add_control('control 3', 💡 )
wn.add_control('control 4', 💡 )

In [ ]:
# Bypass actions and controls
open_action = controls.ControlAction(bypass, 'status', LinkStatus.💡 )
close_action = controls.ControlAction(bypass, 'status', LinkStatus.💡 )

open_ctrl = controls.Control(💡 , 💡 )
close_ctrl = controls.Control(💡 , 💡 )

wn.remove_control('control 5')
wn.remove_control('control 6')
wn.add_control('control 5', open_ctrl)
wn.add_control('control 6', close_ctrl)

print(wn.get_control('control 3'))
print(wn.get_control('control 4'))

In [ ]:
# Run modified simulation using the same network object
sim = wntr.sim.EpanetSimulator(wn)
results_new = sim.run_sim()

## **6. Compare baseline and modified operation**

In [ ]:
# Tank 1
# convert units to ft
tank_level = 💡
tank_level_new = 💡

plt.plot(time_hours, tank_level, color='gray')
plt.plot(time_hours, tank_level_new, color='blue')

plt.xlabel('Time (hours)')
plt.ylabel('Tank level (ft)')
plt.title('Tank 1: baseline and modified controls')
plt.legend(['Baseline', 'Modified'])
plt.show()



In [ ]:
# Pump 335
# convert units to gpm
pump_flow = 💡
pump_flow_new = 💡

plt.plot(time_hours, pump_flow, color='gray')
plt.plot(time_hours, pump_flow_new, color='blue')
plt.xlabel('Time (hours)')
plt.ylabel('Pump flow (gpm)')
plt.title('Pump 335: baseline and modified controls')
plt.legend(['Baseline', 'Modified'])
plt.show()

### **Save the modified network**

WNTR allows us to save changes to an EPANET `.inp` file. This is useful for checking the controls in EPANET or using the modified network later. The file contains the **network and control settings**, not the simulation results.

In [ ]:
# Save the network with the modified event-based controls
wntr.network.write_inpfile(wn, 'Net3_new_event_controls.inp')

💡 **Try it:** Repeat the pump-switching calculation for `results_new`. Did switching decrease? Compare the minimum and maximum tank levels as well.

In [ ]:
# Your turn: calculate pump switching events for results_new


## **7. Time-based controls (explore on your own)**

So far, we have used **event-based controls**, where a pump changes status when the tank level reaches a threshold.

We can also control pumps based on **time**. There are two types:

- **Clock time (daily):** `TimeOfDayCondition` repeats the action at the same time every day.
- **Elapsed simulation time (once):** `SimTimeCondition` triggers the action once at a specified simulation time.

The process is the same as for event-based controls:
1. Define the **action** using `ControlAction`.
2. Define the **condition** using `TimeOfDayCondition` or `SimTimeCondition`.
3. Combine them using `Control`.
4. Remove the old controls and add the new controls.

We will use pump **10** and its existing controls (`control 1` and `control 2`). Reload the original network so the earlier modifications do not carry over.

### **A. At clock time (daily)**

In [ ]:
# Start again with the original network
wn = wntr.network.WaterNetworkModel(inp)
wn.options.time.duration = 48 * 3600

# Check the original controls for pump 10
print(wn.get_control('control 1'))
print(wn.get_control('control 2'))

In [ ]:
# Get pump 10
pump = wn.get_link('10')

# 1. Define actions: turn the pump ON and OFF
open_action = controls.ControlAction(pump, 'status', LinkStatus.Open)
close_action = controls.ControlAction(pump, 'status', LinkStatus.Closed)

# 2. Define conditions: open at 2 AM and close at 6 PM, every day
# Time is expressed in seconds
open_cond = controls.TimeOfDayCondition(wn, '=', 2 * 3600)
close_cond = controls.TimeOfDayCondition(wn, '=', 18 * 3600)

# 3. Combine conditions and actions
open_ctrl = controls.Control(open_cond, open_action)
close_ctrl = controls.Control(close_cond, close_action)

# 4. Replace the existing controls
wn.remove_control('control 1')
wn.remove_control('control 2')
wn.add_control('control 1', open_ctrl)
wn.add_control('control 2', close_ctrl)

# Check the new controls
print(wn.get_control('control 1'))
print(wn.get_control('control 2'))

In [ ]:
# Run the simulation and plot pump 10 status
sim = wntr.sim.EpanetSimulator(wn)
results_time = sim.run_sim()


pump_flow_time = results_time.link['flowrate']['10'] * units
pump_flow = results.link['flowrate']['10'] * units # gpm units

plt.step(time_hours, pump_flow, color = 'gray')
plt.step(time_hours, pump_flow_time, color = 'blue')
plt.xlabel('Time (hours)')
plt.ylabel('Pump 10 status')
plt.title('Daily time-based control')
plt.legend(['Baseline', 'Modified'])
plt.show()

# Save the network
wntr.network.write_inpfile(wn, 'Net3_new_time_controls_daily.inp')

**Try it:** Check when pump 10 opens and closes. Change the times and run the simulation again. Open the saved `.inp` file and find the controls.

### **B. At elapsed simulation time (once)**

Unlike a daily clock-time control, an elapsed-time control is triggered **once**. We will again use pump 10, opening it after 2 hours and closing it after 18 hours.

Reload the original network so the daily controls do not remain active.

In [ ]:
# Reload the original network
wn = wntr.network.WaterNetworkModel(inp)
wn.options.time.duration = 48 * 3600
pump = wn.get_link('10')

# 1. Define actions
open_action = controls.ControlAction(pump, 'status', LinkStatus.Open)
close_action = controls.ControlAction(pump, 'status', LinkStatus.Closed)

# 2. Define conditions: elapsed simulation time in seconds
open_cond = controls.SimTimeCondition(wn, '=', 2 * 3600)
close_cond = controls.SimTimeCondition(wn, '=', 18 * 3600)

# 3. Define controls
open_ctrl = controls.Control(open_cond, open_action)
close_ctrl = controls.Control(close_cond, close_action)

# 4. Replace the existing controls
wn.remove_control('control 1')
wn.remove_control('control 2')
wn.add_control('control 1', open_ctrl)
wn.add_control('control 2', close_ctrl)

# Check and save the controls
print(wn.get_control('control 1'))
print(wn.get_control('control 2'))
wntr.network.write_inpfile(wn, 'Net3_new_time_controls_once.inp')

In [ ]:
# Run the simulation and plot pump 10 status
sim = wntr.sim.EpanetSimulator(wn)
results_once = sim.run_sim()

pump_flow_once = results_once.link['flowrate']['10'] * units
pump_flow = results.link['flowrate']['10'] * units # gpm units

plt.step(time_hours, pump_flow, color = 'gray')
plt.step(time_hours, pump_flow_once, color = 'blue')

plt.xlabel('Time (hours)')
plt.ylabel('Pump 10 status')
plt.title('Elapsed-time control')
plt.show()